In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from pathlib import Path

SEED = 103
rng = np.random.default_rng(SEED)
Faker.seed(SEED)

fake = Faker(["en_US", "fr_FR", "de_DE", "es_ES", "pt_PT", "pl_PL", "es_MX"])
fake_us = Faker(["en_US"])

PAYMENTS_PROVIDERS = ["Paypal", "Stripe", "Mastercard", "Visa", "Bitcoin"]

#----------------------------------------------------------------------------------- Helpers
def make_ids(name: str, n_rows: int) -> list[str]:
    return [f"{name}_{i:05d}" for i in range(1, n_rows + 1)]


def random_dates(start, end, n_rows: int) -> pd.DatetimeIndex:

    s = np.broadcast_to(np.asarray(pd.to_datetime(start), dtype= "datetime64[D]"), n_rows)
    e = np.broadcast_to(np.asarray(pd.to_datetime(end), dtype= "datetime64[D]"), n_rows)
    span = np.maximum((e - s).astype(int), 0)
    offset = (rng.random(n_rows) * (span + 1)).astype(int)
    return pd.to_datetime(s + offset.astype("timedelta64[D]"))



#--------------------------------------------------------------------------------------- Tables

# proprties_df = pd.read_csv("datasets\OLTP_\output\properties.csv")
proprties_df = pd.read_csv("..\\OLTP_\\output\\properties.csv")
rooms_df = pd.read_csv("..\\OLTP_\\output\\rooms.csv")

# booking.com
def generate_bookings_booking_com(properties: pd.DataFrame, rooms: pd.DataFrame, n_rows: int) -> pd.DataFrame:
    
    nights = rng.integers(1, 21, size= n_rows)
    ids = make_ids("booking_com", n_rows)
    birthdates = random_dates("1966-10-01", "2003-10-10", n_rows)
    check_in_dates = random_dates("2018-10-10", "2026-10-10", n_rows)
    check_out_dates = check_in_dates + pd.to_timedelta(nights, unit= "D")

    df = pd.DataFrame({
        "booking_id": ids, 
        "customer_first_name": [fake.first_name() for i in range(n_rows)], 
        "customer_last_name": [fake.last_name() for i in range(n_rows)], 
        "cusstomer_birthday": birthdates, 
        "customer_country": [fake.country() for i in range(n_rows)], 
        "customer_email": [fake.unique.email() for i in range(n_rows)], 
        "customer_phone_number": [fake_us.unique.phone_number() for i in range(n_rows)], 
        "check_in_date": check_in_dates, 
        "check_out_date": check_out_dates,
        "number_of_nights": nights,
        "payment_provider": rng.choice(PAYMENTS_PROVIDERS, size= n_rows),
        "property_id": rng.choice(properties["property_id"], size= n_rows),
        "room_number": rng.choice(rooms["room_number"], size= n_rows),
        "total_amount": np.round(np.abs(rng.normal(50, 500, size= n_rows)), 2)
    })

    # Injecting some Errors

    # Add spend outliers
    idx = df.sample(frac= 0.15).index
    df.loc[idx, "total_amount"] = rng.integers(-76032, 192870, size= len(idx))

    # Add $ to total amount
    currency_idx = df.sample(frac= 0.05).index
    df["total_amount"] = df["total_amount"].astype(str)
    df.loc[currency_idx, "total_amount"] = df.loc[currency_idx, "total_amount"].apply(lambda x: f"${x}")

    # dirty dates end < start
    df.loc[idx, "check_out_date"] = df.loc[idx, "check_in_date"] - pd.to_timedelta(nights[:len(idx)], unit= "D")

    # denormalize values in payment_provider
    df.loc[idx, "payment_provider"] = df.loc[idx, "payment_provider"].map({"Paypal": "payp", "Stripe": "Striipe", "Mastercard": "Master", "Visa": "visa", "Bitcoin": "bit"})
    


    return df

df = generate_bookings_booking_com(proprties_df, rooms_df, 1000)
df.head()


def generate_bookings_airbnb(properties: pd.DataFrame, rooms: pd.DataFrame, n_rows: int) -> pd.DataFrame:
    # airbnb.com

    start_dates = random_dates("2024-10-21", "2026-01-09", n_rows)

    return pd.DataFrame({
    "bking_id": make_ids("airbnb", n_rows), 
    "cust_firstname": [fake.first_name() for i in range(n_rows)], 
    "cust_lastname": [fake.last_name() for i in range(n_rows)], 
    "cust_email": [fake.unique.email() for i in range(n_rows)], 
    "start_date": start_dates, 
    "end_date": start_dates + pd.to_timedelta(rng.integers(1, 15), unit= "D"), 
    "payment_by": rng.choice(PAYMENTS_PROVIDERS, size= n_rows), 
    "property_id": rng.choice(properties["property_id"], size= n_rows), 
    "room_number": rng.choice(rooms["room_id"], size= n_rows), 
    "total_amount": np.round(np.abs(rng.normal(150, 450, size= n_rows)), 2)
    })






def generate_bookings_trip(proprties: pd.DataFrame, rooms: pd.DataFrame, n_rows: int) -> pd.DataFrame:
    # trip.com
    
    nights = rng.integers(1, 21, size= n_rows)
    check_in_dates = random_dates("2025-01-01", "2026-10-01", n_rows)

    return pd.DataFrame({
        "b_id": make_ids("trip", n_rows), 
        "firstname": [fake.first_name() for i in range(n_rows)], 
        "lastname": [fake.last_name() for i in range(n_rows)], 
        "email": [fake.unique.email() for i in range(n_rows)], 
        "check_in_date": check_in_dates, 
        "check_out_date": check_in_dates + pd.to_timedelta(nights, unit= "D"), 
        "property_id": rng.choice(proprties["property_id"], size= n_rows), 
        "room_number": rng.choice(rooms["room_id"], size= n_rows), 
        "amount_paid": rng.integers(150, 950, size= n_rows), 
        "accusation_channel": rng.choice(["Facebook", "Instagram", "Tiktok", "Google Ads", "Youtube"], size= n_rows) 
    })



#------------------------------------------------------------------------------------------------------ Generate tables
def generate_tables(bookings_com_rows: int, airbnb_rows: int, trip_rows: int) -> dict[str, pd.DataFrame]:

    booking_com = generate_bookings_booking_com(proprties_df, rooms_df, bookings_com_rows)
    airbnb = generate_bookings_airbnb(proprties_df, rooms_df, airbnb_rows)
    trip = generate_bookings_trip(proprties_df, rooms_df, trip_rows)

    return {
        "bookingcom": booking_com, 
        "airbnb": airbnb,
        "trip": trip
    }


if __name__ == "__main__":

    tables = generate_tables(75000, 35000, 58500)
    out = Path("output")
    out.mkdir(exist_ok= True)

    for name, df in tables.items():
        df.to_csv(out / f"{name}.csv", index= False)
        print(name, len(df), " rows")



bookingcom 75000  rows
airbnb 35000  rows
trip 58500  rows
